SFT → GRPO: расширенный эксперимент.

В Kaggle включите GPU T4 ×2 и Internet. Добавьте прежний Spider/BIRD, SFT adapter/ + run.json + evaluation/ из sft_full_20261121_103258_artifacts.zip и Secret COMET_API_KEY.

Два независимых запуска от исходного SFT: LR 2.5e-6 и 5e-6, по 512 шагов, пул 5819 вопросов, 2 вопроса × 4 ответа на шаг, микробатч 2, накопление 4. Каждый запуск получает 1024 предъявления вопросов. Пул включает весь train, охват вопросов измеряется отдельно. Это расширенный эксперимент с ограниченным бюджетом.

Сравниваем веса на шагах 128, 256, 512 с исходным SFT на полной validation 956 вопросов. Примерный ориентир по прошлому запуску: около двух часов обучения на каждую карту параллельно, затем последовательная оценка шести моделей. Подготовка vLLM и длинные промпты могут увеличить время.

Адаптеры отправляются в Comet каждые 128 шагов. Снимки сохраняют веса и метаданные для оценки. Состояние оптимизатора для точного resume пока не сохраняется.


In [ ]:
%pip install "vllm==0.31.0" "transformers==5.17.0" "peft==0.21.2" "accelerate==1.15.0" "comet-ml==3.59.0" python-dotenv
%pip install --no-deps --index-url https://download.pytorch.org/whl/cu130 "torchaudio==2.11.0+cu130"
%pip uninstall -y torchao torchcodec
%pip install --no-deps "trl==1.16.0"
%pip install --no-deps --no-cache-dir --index-url https://download.pytorch.org/whl "nvidia-cudnn-cu13==9.23.0.39"

In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from datetime import datetime, timezone

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "text2sql-grpo-sft"
RUN = WORK / "text2sql_runs" / "v1"
BASE = "Qwen/Qwen3.5-2B"
REVISION = "15852e8c16360a2fea060d615a32b45270f8a8fc"

if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/dancravik/text2sql-grpo.git", str(REPO)], check=True)

DATA_DIR = None  # При необходимости укажите Path("/kaggle/input/.../...")
if DATA_DIR is None:
    roots = sorted({
        p.parents[1] for p in INPUT.rglob("train_spider.json")
        if p.parent.name == "spider"
        and (p.parents[1] / "minidev/MINIDEV/mini_dev_sqlite.json").is_file()
    })
    if len(roots) == 1:
        DATA_DIR = roots[0]
    elif roots:
        raise ValueError(f"Укажите DATA_DIR вручную: {roots}")
    else:
        DATA_DIR = WORK / "text2sql_data"
        if not (DATA_DIR / "spider/train_spider.json").is_file():
            archives = list(INPUT.rglob("text2sql-data.zip"))
            if len(archives) != 1:
                raise ValueError(f"Добавьте прежний датасет или задайте DATA_DIR: {archives}")
            with ZipFile(archives[0]) as archive:
                archive.extractall(DATA_DIR)

RUN.mkdir(parents=True, exist_ok=True)
os.environ["DATA_DIR"] = str(DATA_DIR)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.pop("TORCH_CUDNN_V8_API_DISABLED", None)
subprocess.run([sys.executable, "scripts/freeze_splits.py", "--verify"], cwd=REPO, check=True)
print("DATA_DIR:", DATA_DIR)
print("Свободно GiB:", round(shutil.disk_usage(WORK).free / 1024**3, 2))

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["COMET_API_KEY"] = UserSecretsClient().get_secret("COMET_API_KEY")
os.environ["COMET_WORKSPACE"] = "danchenko-rtm"
os.environ["COMET_PROJECT_NAME"] = "text2sql-grpo"

subprocess.run([sys.executable, "-u", "-c", r"""
import importlib.metadata
import torch
from transformers import Trainer
from peft import LoraConfig
from vllm import LLM
from trl import GRPOConfig, GRPOTrainer
for package in ("torch", "transformers", "peft", "accelerate", "vllm", "comet-ml", "nvidia-cudnn-cu13", "trl"):
    print(package, importlib.metadata.version(package))
assert torch.cuda.device_count() == 1
x = torch.randn(1, 6144, 512, device="cuda", dtype=torch.float16, requires_grad=True)
w = torch.randn(6144, 1, 4, device="cuda", dtype=torch.float16)
torch.nn.functional.conv1d(x, w, groups=6144, padding=3).float().mean().backward()
torch.cuda.synchronize()
print("GPU и FP16 conv1d forward/backward OK")
from comet_tracking import start_experiment
experiment = start_experiment(name="sft_grpo_extended_environment_check")
try:
    experiment.log_metric("environment_ready", 1)
finally:
    experiment.end()
"""], cwd=REPO, env={**os.environ, "PYTHONPATH": str(REPO / "src")}, check=True)
gpu_ids = subprocess.check_output(["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"], text=True).split()
assert len(gpu_ids) >= 2, f"Нужны две GPU, найдено {gpu_ids}"
print("GPU для независимых пилотов:", gpu_ids[:2])

Восстановление сохранённого SFT. При нескольких копиях укажите SFT_INPUT как путь к нужной папке в /kaggle/input. Эта ячейка проверяет отчёт и SHA256 весов. Подробная проверка стартовых тензоров выполняется обучающим процессом.

In [ ]:
import hashlib

SFT_INPUT = None  # Например Path("/kaggle/input/datasets/your-name/sft-model")
search_root = Path(SFT_INPUT) if SFT_INPUT is not None else INPUT

def find_sft(root):
    matches = []
    for report_path in root.rglob("run.json"):
        adapter = report_path.parent / "adapter/adapter_model.safetensors"
        if not adapter.is_file():
            continue
        report = json.loads(report_path.read_text())
        if (report.get("status") == "ok" and report.get("num_train_examples") == 5819
                and report.get("global_step") == 728
                and report.get("base_revision") == REVISION):
            matches.append(report_path.parent)
    return sorted(set(matches))

candidates = find_sft(search_root)
if not candidates:
    archives = sorted(search_root.rglob("sft_full_20261121_103258_artifacts.zip"))
    if len(archives) != 1:
        raise ValueError("Добавьте SFT adapter/ и run.json через Add Input. "
                         f"Найдено подходящих ZIP: {archives}")
    restored = WORK / "restored_sft_input"
    restored.mkdir(exist_ok=True)
    with ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            if not (restored / member.filename).resolve().is_relative_to(restored.resolve()):
                raise ValueError("Некорректный путь в ZIP")
        archive.extractall(restored)
    candidates = find_sft(restored)
if len(candidates) != 1:
    raise ValueError(f"Укажите SFT_INPUT с одной копией полного SFT: {candidates}")

SFT = candidates[0]
report = json.loads((SFT / "run.json").read_text())
manifest_hash = hashlib.sha256((REPO / "splits/v1.json").read_bytes()).hexdigest()
assert report["manifest_sha256"] == manifest_hash
assert report["args"]["model"] == BASE and report["optimizer_steps_skipped"] == 0
weights_hash = hashlib.sha256((SFT / "adapter/adapter_model.safetensors").read_bytes()).hexdigest()
print("SFT:", SFT)
print("SHA256 адаптера:", weights_hash)
print("Шагов SFT:", report["global_step"])
if not (SFT / "evaluation/validation.jsonl").is_file():
    raise FileNotFoundError("Входной архив должен содержать evaluation/validation.jsonl для сравнения с SFT")

GROUP = "sft_grpo_extended_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
BUNDLE = RUN / GROUP
print("Новый запуск:", GROUP)

Два обучения идут параллельно, по одному на каждой T4. Все 5819 train-вопросов доступны сэмплеру. В Comet записываются число уникальных использованных вопросов, число предъявлений и состав групп наград. После завершения обоих обучений выполняется последовательная оценка финальных весов.


In [ ]:
if not BUNDLE.exists():
    subprocess.run([
        sys.executable, "-u", "scripts/pilot_grpo_lr.py",
        "--out-dir", str(BUNDLE), "--gpus", *gpu_ids[:2],
        "--init-sft-run", str(SFT), "--lrs", "2.5e-6", "5e-6",
        "--steps", "512", "--micro-batch", "2", "--save-steps", "128", "--keep-checkpoints", "4", "--num-questions", "5819",
        "--dataset", "validation",
    ], cwd=REPO, env=os.environ.copy(), check=True)
else:
    print("Каталог уже есть. Читаем сохранённый результат:", BUNDLE)

selection = json.loads((BUNDLE / "selection.json").read_text())
for row in selection["completed"]:
    print(row["learning_rate"], f"{row['metrics']['greedy_execution_accuracy']:.2%}")
if selection["failed"]:
    print(json.dumps(selection["failed"], indent=2, ensure_ascii=False))
    raise RuntimeError("Есть незавершённые этапы. Последняя ячейка сохранит доступные файлы в ZIP.")

Оценка промежуточных весов после 128 и 256 шагов. Вторая GPU свободна, экспорт и vLLM запускаются по очереди для экономии RAM. Каждая оценка сохраняет свои метрики и отдельный архив в Comet. При повторном запуске готовые варианты пропускаются.


In [ ]:
subprocess.run([
    sys.executable, "-u", "scripts/evaluate_grpo_snapshots.py",
    "--run-dir", str(BUNDLE), "--steps", "128", "256", "--gpu", gpu_ids[0],
], cwd=REPO, env=os.environ.copy(), check=True)
selection = json.loads((BUNDLE / "selection.json").read_text())
assert len(selection["completed"]) == 6


Попарное сравнение с исходным SFT на всех 956 вопросах. Исправленные и потерянные ответы считаются по ID вопроса. Рост train reward оценивается отдельно от точности validation.

In [ ]:
def read_predictions(path):
    records = [json.loads(line) for line in path.open()]
    by_id = {row["id"]: row for row in records}
    assert len(records) == len(by_id) == 956
    assert all(len(row["completions"]) == 1 for row in records)
    return by_id

reference = read_predictions(SFT / "evaluation/validation.jsonl")
def correct(row):
    return row["completions"][0]["outcome"] == "correct"

baseline_count = sum(correct(row) for row in reference.values())
assert baseline_count == 679
comparison = []
for trial in selection["completed"]:
    label = Path(trial["source"]).name
    source = Path(trial["source"])
    training_report_path = source / "training/run.json" if (source / "training/run.json").exists() else source / "run.json"
    training_report = json.loads(training_report_path.read_text())
    evaluation = BUNDLE / "eval" / label
    report = json.loads((evaluation / "metrics.json").read_text())
    assert report["manifest_sha256"] == manifest_hash
    predictions = read_predictions(evaluation / "validation.jsonl")
    assert predictions.keys() == reference.keys()
    for question_id, row in predictions.items():
        assert all(row[key] == reference[question_id][key] for key in ("question", "gold_sql", "db_id"))
    gains = sum(correct(row) and not correct(reference[i]) for i, row in predictions.items())
    losses = sum(not correct(row) and correct(reference[i]) for i, row in predictions.items())
    count = sum(correct(row) for row in predictions.values())
    assert count - baseline_count == gains - losses
    comparison.append({"label": label, "learning_rate": trial["learning_rate"],
                       "global_step": training_report["global_step"],
                       "unique_train_questions": len(training_report.get("seen_example_ids", [])),
                       "question_presentations": training_report.get("question_presentations"),
                       "sft_correct": baseline_count, "grpo_correct": count,
                       "num_questions": len(reference), "gains": gains, "losses": losses,
                       "delta_percentage_points": (count - baseline_count) / len(reference) * 100,
                       "metrics": report["datasets"]["validation"]})
    print(label, f"Accuracy {count / len(reference):.2%}; исправлено {gains}; потеряно {losses}")
(BUNDLE / "comparison_vs_sft.json").write_text(json.dumps(comparison, indent=2) + "\n")
print("\nВсе варианты по точности validation:")
for row in sorted(comparison, key=lambda item: item["grpo_correct"], reverse=True):
    print(row["label"], f"{row['grpo_correct'] / 956:.2%}",
          "уникальных train-вопросов", row["unique_train_questions"])


Итоговый архив можно создать и после ошибки обучения. Внутри будут доступные адаптеры, метрики, ответы, снимки адаптеров, логи, исходный SFT и код. Финальные GRPO-адаптеры и промежуточные снимки также отправляются в Comet во время работы. Скачайте ZIP через ссылку или Files.

In [ ]:
ARCHIVE = WORK / f"{BUNDLE.name}_complete.zip"
with ZipFile(ARCHIVE, "w", ZIP_DEFLATED, allowZip64=True) as archive:
    all_evaluated = (BUNDLE / "comparison_vs_sft.json").exists() and len(json.loads((BUNDLE / "selection.json").read_text()).get("completed", [])) == 6
    for path in sorted(BUNDLE.rglob("*")):
        relative = path.relative_to(BUNDLE)
        if all_evaluated and "trainer" in relative.parts:
            continue  # Evaluated snapshots already have copies in snapshots/.
        if path.is_file() and path.suffix != ".zip" and "merged" not in relative.parts:
            archive.write(path, BUNDLE.name + "/" + relative.as_posix())
    for relative in [Path("run.json"), *[p.relative_to(SFT) for p in (SFT / "adapter").rglob("*") if p.is_file()],
                     Path("evaluation/metrics.json"), Path("evaluation/validation.jsonl")]:
        path = SFT / relative
        if path.is_file():
            archive.write(path, "initial_sft/" + relative.as_posix())
    for folder in ("src", "scripts"):
        for path in (REPO / folder).rglob("*.py"):
            archive.write(path, "repository/" + path.relative_to(REPO).as_posix())
    archive.write(REPO / "splits/v1.json", "repository/splits/v1.json")
    archive.writestr("environment/pip_freeze.txt", subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
with ZipFile(ARCHIVE) as archive:
    assert archive.testzip() is None
from IPython.display import FileLink, display
print(ARCHIVE, f"{ARCHIVE.stat().st_size / 1024**2:.1f} MiB")
display(FileLink(ARCHIVE.relative_to(WORK).as_posix()))